In [ ]:
import os
import csv
from datadis_python.client.v2.simple_client import SimpleDatadisClientV2

# ==========================================
# CONFIGURACIÓN Y CREDENCIALES
# ==========================================
nif = "22958277E"
password = "Tfm_2026%"
nombre_archivo = "consumo_2anios_final.csv"

# ==========================================
# PASO 1: COMPROBACIÓN DE SEGURIDAD
# ==========================================
if os.path.exists(nombre_archivo):
    print(f"El archivo '{nombre_archivo}' ya existe. No llamamos a la API para evitar bloqueos.")
else:
    try:
        print("1. Conectando con el cliente oficial de Datadis...")
        
        with SimpleDatadisClientV2(nif, password) as client:
            
            print("Autenticando y obteniendo datos del suministro con la distribuidora...")
            supplies_response = client.get_supplies()
            
            if not supplies_response.supplies:
                print("Error: No se encontraron suministros asociados a este NIF.")
            else:
                # Obtenemos el primer suministro disponible
                supply = supplies_response.supplies[0]
                
                # Guardamos los datos REALES de 20 caracteres que Iberdrola sí reconoce
                cups_real_distribuidora = supply.cups.strip() 
                distribuidora_real = supply.distributor_code
                
                print("¡Autenticación exitosa con la distribuidora!")
                print(f"-> CUPS real detectado (20 caracteres): {cups_real_distribuidora}")
                print(f"-> Código de distribuidora: {distribuidora_real}")
                
                print("\n2. Descargando curva de consumo...")
                
                # ==========================================
                # SOLUCIÓN DEFINITIVA: CUPS REAL + PUNTO DE MEDIDA
                # ==========================================
                # Enviamos el CUPS de 20 caracteres LIMPIO (para que Iberdrola no falle)
                # y le sumamos el 'point_profile' para cumplir con los requisitos de la API.
                respuesta_consumo = client.get_consumption(
                    cups=cups_real_distribuidora,        # <-- El de 20 caracteres, sin inventar letras
                    distributor_code=distribuidora_real, # Código 82 (Iberdrola)
                    date_from="2024/06",
                    date_to="2026/05",
                    measurement_type=0,                  # 0 = Consumo
                    point_type=5,                        # 5 = Residencial
                    #point_profile="1P",                  # <-- Indica que es el contador principal
                    #authorized_nif=nif                   
                )
                
                # ==========================================
                # PASO 4: PROCESAMIENTO Y VOLCADO A CSV
                # ==========================================
                datos = []
                
                # Ahora sabemos que la librería guarda los registros en 'time_curve'
                if hasattr(respuesta_consumo, 'time_curve') and respuesta_consumo.time_curve:
                    datos = respuesta_consumo.time_curve
                
                if datos:
                    print(f"¡Éxito total! Se han detectado {len(datos)} registros de consumo listos para guardar.")
                    
                    with open(nombre_archivo, mode="w", newline="", encoding="utf-8") as file:
                        writer = csv.writer(file, delimiter=";")
                        # Escribimos las cabeceras del CSV
                        writer.writerow(["CUPS", "Fecha", "Hora", "Consumo (kWh)"])
                        
                        for reg in datos:
                            # Al ser un modelo de Pydantic, extraemos sus atributos de forma segura
                            fecha = getattr(reg, 'date', '')
                            hora = getattr(reg, 'time', '')
                            # Por si la librería lo llama 'consumption' o 'consumption_kwh'
                            consumo = getattr(reg, 'consumption', getattr(reg, 'consumption_kwh', 0.0))
                            
                            # Guardamos en el archivo final
                            writer.writerow([cups_real_distribuidora, fecha, hora, consumo])
                            
                    print(f"¡PROCESO COMPLETADO CON ÉXITO! Archivo guardado como: {nombre_archivo}")
                else:
                    print("Error: La variable 'time_curve' está vacía. No hay datos en esas fechas.")
                    
    except Exception as e:
        print(f"Fallo crítico en la ejecución: {e}")

1. Conectando con el cliente oficial de Datadis...
Autenticando y obteniendo datos del suministro con la distribuidora...
Obteniendo lista de suministros...
Autenticando con Datadis...
Autenticación exitosa
Petición a /get-supplies-v2 (intento 1/4)...
Respuesta exitosa (845 chars)
1 suministros validados
Advertencia: 2 errores de distribuidor
¡Autenticación exitosa con la distribuidora!
-> CUPS real detectado (20 caracteres): ES0021000005946311VA
-> Código de distribuidora: 8

2. Descargando curva de consumo...
Obteniendo consumo para ES0021000005946311VA (2024/06 - 2026/05)...
Petición a /get-consumption-data-v2 (intento 1/4)...
Respuesta exitosa (4280469 chars)
17062 registros de consumo validados
¡Éxito total! Se han detectado 17062 registros de consumo listos para guardar.
¡PROCESO COMPLETADO CON ÉXITO! Archivo guardado como: consumo_2anios_final.csv
